# Evaluación de la Red Siamesa

Este notebook ejecuta el script de evaluación y analiza las métricas del modelo entrenado.

**Notebooks previos requeridos:**
- `01_dataset_preparation` — prepara los frames y recorta las caras válidas.
- `02_pair_generation_preview` — genera los pares siameses de entrenamiento, validación y test.
- `03_siamese_model_summary` — describe la arquitectura del modelo.
- `04_training_results` — ejecuta el entrenamiento y analiza las curvas.

Este notebook **no define** métricas manualmente ni entrena el modelo.
Delega la evaluación en `src/evaluation/evaluate.py` y luego visualiza los artefactos generados.

## Parámetros

In [ ]:
# --- Parámetros editables ---

# False por defecto: evita ejecutar la evaluación antes de que el modelo esté listo.
# Cambia a True cuando quieras ejecutar la evaluación completa.
RUN_EVALUATION = False

MODEL_NAME = "siamese_model.keras"

# Si no es None, se usa esta ruta directa en lugar de buscar por MODEL_NAME.
MODEL_PATH = None

BATCH_SIZE = 32

# Umbral de similitud para clasificar un par como GRANTED.
# Valor por defecto: 0.5 (coincide con DEFAULT_SIMILARITY_THRESHOLD en src.config).
THRESHOLD = 0.5

# Si True, muestra una tabla de muestras de predicción al final del notebook.
SHOW_PREDICTION_SAMPLES = False
NUM_PREDICTION_SAMPLES = 10

## Importaciones y configuración

In [ ]:
import sys
import subprocess
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image as IPyImage

# Detectar la raíz del proyecto sin importar desde qué directorio se abre el notebook
_cwd = Path.cwd()
if (_cwd / "src").exists():
    project_root = _cwd
elif (_cwd.parent / "src").exists():
    project_root = _cwd.parent
else:
    raise RuntimeError(
        f"No se encontró el directorio src/. Directorio actual: {_cwd}"
    )

sys.path.insert(0, str(project_root))

from src.config import (
    PROJECT_ROOT,
    PAIRS_DIR,
    SAVED_MODEL_DIR,
    METRICS_DIR,
    PLOTS_DIR,
    DEFAULT_SIMILARITY_THRESHOLD,
)

print(f"Python              : {sys.executable}")
print(f"project_root        : {project_root}")
print(f"PAIRS_DIR           : {PAIRS_DIR}")
print(f"SAVED_MODEL_DIR     : {SAVED_MODEL_DIR}")
print(f"METRICS_DIR         : {METRICS_DIR}")
print(f"PLOTS_DIR           : {PLOTS_DIR}")
print(f"THRESHOLD activo    : {THRESHOLD}")
print(f"DEFAULT_THRESHOLD   : {DEFAULT_SIMILARITY_THRESHOLD}")

## Función auxiliar

In [ ]:
def run_command(command: list[str]) -> None:
    """Ejecuta un comando de shell e imprime stdout y stderr."""
    print("Comando:", " ".join(command))
    print("-" * 60)
    result = subprocess.run(
        command,
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("[stderr]")
        print(result.stderr)
    print("-" * 60)
    print(f"Código de retorno: {result.returncode}")

## Verificación de prerequisitos

Comprueba que los archivos necesarios existan antes de ejecutar la evaluación.

In [ ]:
test_csv   = PAIRS_DIR / "test_pairs.csv"
model_path = Path(MODEL_PATH) if MODEL_PATH else SAVED_MODEL_DIR / MODEL_NAME

test_csv_ok = test_csv.exists()
model_ok    = model_path.exists()

if not test_csv_ok:
    print("Prerequisito faltante: test_pairs.csv no encontrado.")
    print("  Primero ejecuta el notebook 02 o el comando:")
    print("    python -m src.dataset.build_pairs --overwrite")
    print()

if not model_ok:
    print(f"Prerequisito faltante: modelo no encontrado en {model_path}")
    print("  Primero ejecuta el notebook 04 o el comando:")
    print("    python -m src.training.train")
    print()

if test_csv_ok and model_ok:
    print("Prerequisitos verificados.")
    print(f"  test_pairs.csv : {test_csv}")
    print(f"  modelo         : {model_path}")

## Resumen del conjunto de test

Distribución de pares positivos y negativos en el split de test.

In [ ]:
if test_csv_ok:
    test_df  = pd.read_csv(test_csv)
    total    = len(test_df)
    positive = int((test_df["label"] == 1).sum())
    negative = int((test_df["label"] == 0).sum())

    print(f"Total de pares de test     : {total}")
    print(f"  Positivos (mismo usuario): {positive}")
    print(f"  Negativos (distintos)    : {negative}")
    print()

    # Distribución por par de vistas (columnas view_a / view_b del CSV de pares)
    test_df["view_pair"] = test_df["view_a"] + " / " + test_df["view_b"]
    view_dist = (
        test_df.groupby(["view_pair", "label"])
        .size()
        .reset_index(name="count")
    )
    display(view_dist)
else:
    print("test_pairs.csv no disponible. Ejecuta primero la celda de prerequisitos.")

## Comando de evaluación

Si `RUN_EVALUATION = True`, lanza `src/evaluation/evaluate.py` con los parámetros definidos arriba.

Si `RUN_EVALUATION = False`, imprime el comando que se ejecutaría sin lanzarlo.

In [ ]:
eval_command = [
    sys.executable, "-m", "src.evaluation.evaluate",
    "--model-name",  MODEL_NAME,
    "--batch-size",  str(BATCH_SIZE),
    "--threshold",   str(THRESHOLD),
]

if MODEL_PATH is not None:
    eval_command += ["--model-path", str(MODEL_PATH)]

if RUN_EVALUATION:
    run_command(eval_command)
else:
    print("RUN_EVALUATION = False — evaluación no ejecutada.")
    print()
    print("Comando que se ejecutaría:")
    print(" ", " ".join(eval_command))

## Estado de los artefactos generados

Rutas esperadas después de la evaluación y estado actual en disco.

In [ ]:
report_path           = METRICS_DIR / "evaluation_report.json"
predictions_path      = METRICS_DIR / "test_predictions.csv"
confusion_matrix_path = PLOTS_DIR   / "confusion_matrix.png"
roc_curve_path        = PLOTS_DIR   / "roc_curve.png"

artifacts = [
    ("Reporte de evaluación",     report_path),
    ("Predicciones de test",      predictions_path),
    ("Matriz de confusión (png)", confusion_matrix_path),
    ("Curva ROC (png)",           roc_curve_path),
]

status_rows = [
    {
        "Artefacto": label,
        "Ruta":      str(path),
        "Existe":    path.exists(),
    }
    for label, path in artifacts
]
display(pd.DataFrame(status_rows))

## Reporte de evaluación

Métricas generadas por el script de evaluación.

In [ ]:
METRIC_LABELS = {
    "threshold":     "Umbral",
    "num_samples":   "Muestras",
    "num_positives": "Positivos",
    "num_negatives": "Negativos",
    "accuracy":      "Accuracy",
    "precision":     "Precision",
    "recall":        "Recall",
    "f1_score":      "F1 Score",
    "far":           "FAR",
    "frr":           "FRR",
    "roc_auc":       "ROC AUC",
}

report = None

if report_path.exists():
    with open(report_path, "r", encoding="utf-8") as f:
        report = json.load(f)

    metric_rows = [
        {"Métrica": METRIC_LABELS.get(k, k), "Valor": v}
        for k, v in report.items()
        if k in METRIC_LABELS
    ]
    display(pd.DataFrame(metric_rows))
else:
    print("El reporte de evaluación no existe todavía.")
    print("Cambia RUN_EVALUATION = True y vuelve a ejecutar la celda de evaluación.")

## Matriz de confusión

Valores de TN, FP, FN y TP obtenidos con el umbral de evaluación.

In [ ]:
if report is not None:
    cm_data = report.get("confusion_matrix", {})
    tn = cm_data.get("true_negative",  0)
    fp = cm_data.get("false_positive", 0)
    fn = cm_data.get("false_negative", 0)
    tp = cm_data.get("true_positive",  0)

    cm_df = pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=pd.Index(["Real: DENIED (0)", "Real: GRANTED (1)"], name=""),
        columns=["Pred: DENIED (0)", "Pred: GRANTED (1)"],
    )
    display(cm_df)
else:
    print("No hay datos de matriz de confusión. Ejecuta la evaluación primero.")

## Gráficas guardadas

Visualizaciones generadas por el script de evaluación.

In [ ]:
for plot_path, title in [
    (confusion_matrix_path, "Matriz de confusión"),
    (roc_curve_path,        "Curva ROC"),
]:
    if plot_path.exists():
        print(f"--- {title} ---")
        display(IPyImage(filename=str(plot_path)))
    else:
        print(f"{title}: gráfica no disponible. Ejecuta la evaluación primero.")

## Predicciones del modelo

Análisis de los puntajes de similitud generados para cada par del conjunto de test.

In [ ]:
predictions_df = None

if predictions_path.exists():
    predictions_df = pd.read_csv(predictions_path)

    print(f"Total de predicciones: {len(predictions_df)}")
    print()
    print("Primeras filas:")
    display(predictions_df.head(10))

    print()
    print("Resumen estadístico de y_score:")
    display(predictions_df["y_score"].describe().to_frame())

    # Histograma de puntajes agrupado por clase real
    fig, ax = plt.subplots()
    for label, group in predictions_df.groupby("y_true"):
        ax.hist(
            group["y_score"],
            bins=30,
            alpha=0.6,
            label=f"y_true = {int(label)}",
        )
    ax.axvline(x=THRESHOLD, linestyle="--", label=f"Umbral = {THRESHOLD}")
    ax.set_title("Distribución de puntajes de similitud")
    ax.set_xlabel("y_score")
    ax.set_ylabel("Frecuencia")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("test_predictions.csv no disponible. Ejecuta la evaluación primero.")

## Análisis de umbral

Impacto del umbral de similitud en las métricas de seguridad y experiencia de usuario.
Este análisis es solo para presentación; no sobreescribe `evaluation_report.json`.

In [ ]:
if predictions_df is not None:
    thresholds  = [round(t / 10, 1) for t in range(1, 10)]
    y_true_arr  = predictions_df["y_true"].values
    y_score_arr = predictions_df["y_score"].values

    threshold_rows = []
    for t in thresholds:
        y_pred_t = (y_score_arr >= t).astype(int)

        # Conteo de cada caso de la matriz de confusión para este umbral
        tn_t = int(((y_pred_t == 0) & (y_true_arr == 0)).sum())
        fp_t = int(((y_pred_t == 1) & (y_true_arr == 0)).sum())
        fn_t = int(((y_pred_t == 0) & (y_true_arr == 1)).sum())
        tp_t = int(((y_pred_t == 1) & (y_true_arr == 1)).sum())

        acc_t = (tp_t + tn_t) / len(y_true_arr) if len(y_true_arr) > 0 else 0.0
        far_t = fp_t / (fp_t + tn_t) if (fp_t + tn_t) > 0 else 0.0
        frr_t = fn_t / (fn_t + tp_t) if (fn_t + tp_t) > 0 else 0.0

        threshold_rows.append({
            "Umbral":   t,
            "Accuracy": round(acc_t, 4),
            "FAR":      round(far_t, 4),
            "FRR":      round(frr_t, 4),
        })

    threshold_df = pd.DataFrame(threshold_rows)
    display(threshold_df)

    # Gráfica FAR y FRR vs umbral
    fig, ax = plt.subplots()
    ax.plot(threshold_df["Umbral"], threshold_df["FAR"], marker="o", label="FAR")
    ax.plot(threshold_df["Umbral"], threshold_df["FRR"], marker="s", label="FRR")
    ax.set_title("FAR y FRR según el umbral de similitud")
    ax.set_xlabel("Umbral")
    ax.set_ylabel("Tasa de error")
    ax.set_xticks(thresholds)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("test_predictions.csv no disponible. Ejecuta la evaluación primero.")

## Muestras de predicción (opcional)

Controlado por `SHOW_PREDICTION_SAMPLES`. No muestra imágenes de caras.

In [ ]:
if SHOW_PREDICTION_SAMPLES:
    if predictions_df is not None:
        sample_df = predictions_df[["y_true", "y_score", "y_pred"]].head(NUM_PREDICTION_SAMPLES)
        display(sample_df)
    else:
        print("test_predictions.csv no disponible.")
else:
    print("Visualización de muestras desactivada (SHOW_PREDICTION_SAMPLES = False).")

## Interpretación de métricas

- **Accuracy**: proporción de predicciones correctas sobre el total de pares.
- **Precision**: de los pares clasificados como GRANTED, qué fracción era realmente genuina.
- **Recall**: de todos los pares genuinos, qué fracción fue correctamente aceptada.
- **FAR** (*False Acceptance Rate*): proporción de impostores aceptados — `FP / (FP + TN)`.  
  Un FAR bajo es crítico para la seguridad del sistema.
- **FRR** (*False Rejection Rate*): proporción de usuarios genuinos rechazados — `FN / (FN + TP)`.  
  Un FRR bajo mejora la experiencia del usuario.
- **ROC AUC**: capacidad discriminativa del modelo independientemente del umbral.  
  Un valor de 1.0 indica separación perfecta; 0.5 equivale a clasificación aleatoria.
- **Umbral**: controla el balance entre seguridad (FAR) y conveniencia (FRR).  
  El punto donde FAR ≈ FRR se denomina *Equal Error Rate* (EER).

## Lista de verificación

- [ ] Modelo entrenado y guardado en `models/saved_model/`.
- [ ] Pares de test disponibles (`data/pairs/test_pairs.csv`).
- [ ] Evaluación ejecutada (`RUN_EVALUATION = True`).
- [ ] Reporte generado (`outputs/metrics/evaluation_report.json`).
- [ ] Matriz de confusión revisada.
- [ ] FAR y FRR analizados.
- [ ] Umbral seleccionado según el balance seguridad/conveniencia.
- [ ] **Outputs limpios antes del commit** (`Kernel → Restart & Clear Output`).